In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Install dependencies
!pip install sentence-transformers qdrant-client tqdm torch

In [3]:
import json
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import gc

# --- CONFIGURATION ---
CORPUS_FILE = "/content/drive/MyDrive/chunked_rag_corpus.json"
VECTOR_OUTPUT = "/content/drive/MyDrive/loksabha_vectors.npy"


In [4]:
print(f"Loading {CORPUS_FILE}...")
with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    records = json.load(f)

Loading /content/drive/MyDrive/chunked_rag_corpus.json...


In [5]:
texts_to_embed = []
for doc in records:
    meta = doc["metadata"]
    doc_type = meta.get("type", "Q&A").upper()
    title = meta.get("title", "Unknown")
    texts_to_embed.append(f"Type: {doc_type} | Subject: {title}\n\n{doc['raw_text']}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BAAI/bge-m3 on: {device.upper()} in FP16...")

# --- OPTIMIZATION 1: FP16 TENSOR CORES ---
# This halves memory usage and massively accelerates matrix math on the T4 GPU
model = SentenceTransformer(
    "BAAI/bge-m3",
    device=device,
    model_kwargs={"torch_dtype": torch.float16}
)

# --- OPTIMIZATION 2: TIGHTER GUILLOTINE ---
# 400 words rarely exceeds 600 tokens. Capping at 800 guarantees zero wasted math.
model.max_seq_length = 800

# Clear out any lingering garbage in the RAM
gc.collect()
torch.cuda.empty_cache()

print(f"Starting max-speed FP16 generation for {len(texts_to_embed)} chunks...")

# --- OPTIMIZATION 3: DOUBLED BATCH SIZE ---
# Because FP16 uses half the memory, we can safely slam 64 documents into the GPU at once
vectors = model.encode(texts_to_embed, batch_size=64, show_progress_bar=True)

print(f"Saving vectors to {VECTOR_OUTPUT}...")
np.save(VECTOR_OUTPUT, vectors)
print("Done! You can close Google Colab now.")

Loading BAAI/bge-m3 on: CUDA in FP16...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Starting max-speed FP16 generation for 168961 chunks...


Batches:   0%|          | 0/2641 [00:00<?, ?it/s]

Saving vectors to /content/drive/MyDrive/loksabha_vectors.npy...
Done! You can close Google Colab now.
